# Image I/O, Formats, and Metadata

> **Beginner · Image fundamentals**


## Why this matters

A pixel array is only part of an image file. Compression, bit depth, orientation, and metadata affect fidelity and reproducibility.

**Where it appears:** Dataset ingestion, web export, camera-photo processing, quality checks, and resilient batch loaders.


## Learning Objectives

- Understand lossy vs lossless compression and when to use each format
- Control JPEG/PNG encoding parameters explicitly
- Measure the practical size/quality trade-off instead of guessing
- Read/write images robustly, including unicode paths and unusual bit depths
- Extract EXIF metadata (orientation, camera settings) using Pillow alongside OpenCV
- Build a batch image-loading pipeline that skips corrupted files gracefully


## Prerequisites

03 OpenCV Setup and Your First Pipeline

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.imread`, `cv2.imwrite`, `cv2.imencode`, Pillow `Image.open`, EXIF access

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Image Representation and File Formats

File format choice is a real engineering decision, not a default to accept.
**PNG** is lossless and supports transparency (good for masks, diagrams,
anything with hard edges/text). **JPEG** is lossy and much smaller for
natural photographs, but repeated re-encoding introduces cumulative
artifacts. **WebP** often beats both for web delivery. This notebook
measures the trade-off directly instead of asserting it.


### Image I/O and Metadata

`cv2.imread`/`cv2.imwrite` handle pixel data only -- no EXIF metadata, and
historically unreliable with non-ASCII file paths on some platforms. Real
pipelines therefore often pair OpenCV (fast pixel operations) with Pillow
(`PIL.Image`) purely for metadata and safer path handling, then convert to
a NumPy array for OpenCV processing.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Image Representation and File Formats


### 1. Encoding to different formats in memory

`cv2.imencode` compresses an array to an in-memory buffer without touching disk -- useful for measuring size or streaming over a network.


In [ ]:
import cv2
import numpy as np


def encode_sizes(image: np.ndarray) -> dict:
    """Encode the same image at different format/quality settings and report byte sizes."""
    results = {}
    ok, buf = cv2.imencode(".png", image)
    results["PNG (lossless)"] = len(buf)
    for quality in (95, 75, 30):
        ok, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, quality])
        results[f"JPEG q={quality}"] = len(buf)
    return results


# Let's use a real photograph (mountain landscape)
photo = load_real_image("images/landscapes", "mountain.jpg")
print("File sizes for a natural photograph:")
for fmt, size in encode_sizes(photo).items():
    print(f"{fmt:16s}: {size:8d} bytes")

### 2. Measuring lossy artifacts quantitatively

JPEG compression is designed for natural photos, where human eyes are less sensitive to high-frequency color changes. However, on images with sharp edges (like text, diagrams, or UI screenshots), JPEG introduces terrible 'ringing' artifacts. Let's quantitatively measure this difference by comparing JPEG degradation on a photo vs. a document.


In [ ]:
def jpeg_roundtrip_error(image: np.ndarray, quality: int) -> float:
    """Encode at `quality`, decode, and return the mean absolute pixel difference."""
    ok, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, quality])
    decoded = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    diff = cv2.absdiff(image, decoded)
    return float(diff.mean())


# Load a document/receipt image
document = load_real_image("images/documents", "text.png")

print("Mean Absolute Pixel Error (higher is worse):")
print(f"{'Quality':<10} | {'Photograph':<12} | {'Document/Text':<12}")
print("-" * 40)
for q in (95, 75, 50, 20):
    err_photo = jpeg_roundtrip_error(photo, q)
    err_doc = jpeg_roundtrip_error(document, q)
    print(f"{q:<10} | {err_photo:<12.3f} | {err_doc:<12.3f}")

print(
    "\nNotice how the error is often significantly higher on the sharp document image! Always use PNG for text/diagrams."
)

### 3. Saving and re-reading with metadata considerations

`cv2.imwrite`/`cv2.imread` round-trip through disk. Note that OpenCV does **not** preserve EXIF metadata on write -- that limitation is covered fully in the next notebook (Image I/O and Metadata).


In [ ]:
from pathlib import Path
from cv_utils import ensure_dir


def save_and_reload(image: np.ndarray, out_dir: str, name: str) -> np.ndarray:
    out_dir = ensure_dir(out_dir)
    path = out_dir / name
    cv2.imwrite(str(path), image)
    reloaded = cv2.imread(str(path))
    assert reloaded is not None, f"Failed to reload {path}"
    return reloaded


# PNG is lossless, so round-tripping should yield the exact same pixel matrix
reloaded = save_and_reload(photo, "outputs_04", "photo_copy.png")
print("Round-trip identical (PNG):", np.array_equal(photo, reloaded))

## Part 2: Image I/O and Metadata


### 1. Robust image loading for a folder of files

A batch loader must not crash on the first corrupted/unsupported file -- it should report failures and continue, which is what real dataset pipelines require.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, ensure_dir


def load_folder(folder: str) -> tuple[dict[str, np.ndarray], list[str]]:
    """Load every image in `folder`. Returns (successful_images, failed_filenames)."""
    folder = Path(folder)
    loaded, failed = {}, []
    for path in sorted(folder.glob("*")):
        img = cv2.imread(str(path))
        if img is None:
            failed.append(path.name)
        else:
            loaded[path.name] = img
    return loaded, failed


# Build a small demo folder: 2 valid PNGs + 1 corrupted "image"
demo_dir = ensure_dir("outputs_05/batch")
cv2.imwrite(str(demo_dir / "a.png"), load_real_image("images/objects", "coins.jpg"))
cv2.imwrite(str(demo_dir / "b.png"), load_real_image("images/documents", "text.png"))
(demo_dir / "corrupt.png").write_bytes(b"not a real png")

loaded, failed = load_folder(demo_dir)
print("Loaded:", list(loaded.keys()))
print("Failed:", failed)


### 2. Reading EXIF metadata with Pillow

OpenCV ignores EXIF. Use Pillow to read it (falls back gracefully if a file has none, which is the case for our synthetic PNGs).


In [ ]:
from PIL import Image
from PIL.ExifTags import TAGS


def read_exif(path: str) -> dict:
    """Return a dict of human-readable EXIF tags, or {} if none exist."""
    with Image.open(path) as im:
        raw = im.getexif()
        if not raw:
            return {}
        return {TAGS.get(tag_id, tag_id): value for tag_id, value in raw.items()}


exif = read_exif(demo_dir / "a.png")
print("EXIF tags found:", exif if exif else "(none -- expected for a synthetic PNG)")

### 3. Bridging Pillow and OpenCV correctly

When metadata handling matters, load with Pillow, then convert to an OpenCV-compatible NumPy array -- remembering the RGB->BGR channel swap.


In [ ]:
def pil_to_opencv(pil_image: "Image.Image") -> np.ndarray:
    """Convert a Pillow RGB image to an OpenCV-style BGR NumPy array."""
    rgb_array = np.array(pil_image.convert("RGB"))
    return cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)


with Image.open(demo_dir / "a.png") as im:
    cv_image = pil_to_opencv(im)

print("Converted shape/dtype:", cv_image.shape, cv_image.dtype)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Image Representation and File Formats: Analyzing Image Quality vs File Size

When deploying models or streaming video, we must balance compression artifacts against network bandwidth. Here we compress an image at varying levels, record the file size in bytes, and compute Peak Signal-to-Noise Ratio (PSNR) to measure degradation quantitatively.


In [ ]:
import io


def analyze_jpeg_compression(image: np.ndarray, label: str):
    qualities = [10, 30, 50, 70, 90, 100]
    results = []
    print(f"\n--- PSNR Analysis for {label} ---")
    for q in qualities:
        # Encode to in-memory buffer
        success, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, q])
        if not success:
            continue

        # Measure compressed size
        size_bytes = len(buf)

        # Decode back to calculate error
        decoded = cv2.imdecode(buf, cv2.IMREAD_COLOR)

        # Compute Peak Signal-to-Noise Ratio (PSNR)
        mse = np.mean((image.astype(np.float32) - decoded.astype(np.float32)) ** 2)
        psnr = 20 * np.log10(255.0 / np.sqrt(mse)) if mse > 0 else float("inf")

        results.append((q, size_bytes, psnr))
        print(
            f"JPEG Quality: {q:3d} | Size: {size_bytes:8d} bytes | PSNR: {psnr:.2f} dB"
        )

    return results


results_photo = analyze_jpeg_compression(photo, "Mountain Photo")

### Mini Project — Image I/O and Metadata: Multi-Threaded Batch Image Preprocessor

In deep learning applications, loading and resizing folders of images can form a bottleneck on the CPU. Here, we build a multi-threaded batch loader that reads and resizes images concurrently, maximizing CPU throughput.


In [ ]:
import concurrent.futures
from pathlib import Path

# Create a mock folder with a few image files
img_dir = Path("demo_images")
img_dir.mkdir(exist_ok=True)
sample_img = load_real_image("images/objects", "coins.jpg")
for i in range(5):
    cv2.imwrite(str(img_dir / f"test_{i}.jpg"), sample_img)


def process_single_image(path: Path, target_size=(128, 128)) -> tuple[str, np.ndarray]:
    # Thread-safe read and resize
    img = cv2.imread(str(path))
    resized = cv2.resize(img, target_size)
    return path.name, resized


def parallel_batch_loader(folder: Path, num_threads=4) -> dict[str, np.ndarray]:
    image_paths = list(Path(folder).glob("*.jpg"))
    results = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        future_to_path = {
            executor.submit(process_single_image, p): p for p in image_paths
        }
        for future in concurrent.futures.as_completed(future_to_path):
            try:
                name, data = future.result()
                results[name] = data
            except Exception as e:
                print(f"Error reading image: {e}")

    return results


batch = parallel_batch_loader(img_dir)
print(f"Loaded and resized {len(batch)} images in parallel.")
# Clean up files
for p in img_dir.glob("*.jpg"):
    p.unlink()
img_dir.rmdir()

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Image Representation and File Formats
1. Plot bytes-vs-quality for JPEG quality levels 10 to 100 in steps of 10 using matplotlib.
2. Compare the degradation (e.g. using RMSE) of an image with sharp edges (checkerboard) vs smooth gradients when applying JPEG compression.
3. Write `best_quality_under_budget(image, max_bytes)` that searches for the highest JPEG quality fitting a byte budget.

Use the empty cell below to work through them.



#### Solutions — Image Representation and File Formats

In [ ]:
# Solution 1: Plot bytes-vs-quality for JPEG quality levels 10 to 100
def plot_jpeg_tradeoff(image: np.ndarray) -> None:
    qualities = list(range(10, 101, 10))
    sizes = []
    for q in qualities:
        _, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, q])
        sizes.append(len(buf))

    plt.figure(figsize=(6, 4))
    plt.plot(qualities, sizes, "o-", color="purple")
    plt.title("JPEG Quality vs. Encoded File Size")
    plt.xlabel("Quality Level")
    plt.ylabel("Size (Bytes)")
    plt.grid(True)
    plt.show()

In [ ]:
# Solution 2: Compare degradation of checkerboard vs smooth gradient
import cv2
import numpy as np

# Smooth gradient image
smooth_img = np.tile(np.linspace(0, 255, 300, dtype=np.uint8), (300, 1))
# Sharp edges checkerboard image
sharp_img = np.zeros((300, 300), dtype=np.uint8)
for y in range(0, 300, 30):
    for x in range(0, 300, 30):
        if ((x // 30) + (y // 30)) % 2 == 0:
            sharp_img[y : y + 30, x : x + 30] = 255


def rmse_error(img1, img2):
    return np.sqrt(np.mean((img1.astype(np.float32) - img2.astype(np.float32)) ** 2))


def compare_degradation(smooth, sharp, quality=50):
    # Smooth roundtrip
    _, buf_s = cv2.imencode(".jpg", smooth, [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec_s = cv2.imdecode(buf_s, cv2.IMREAD_GRAYSCALE)

    # Sharp roundtrip
    _, buf_h = cv2.imencode(".jpg", sharp, [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec_h = cv2.imdecode(buf_h, cv2.IMREAD_GRAYSCALE)

    print(f"RMSE Smooth (Gradient): {rmse_error(smooth, dec_s):.2f}")
    print(f"RMSE Sharp (Checkerboard): {rmse_error(sharp, dec_h):.2f}")
    # Explanation: Sharp checkerboards degrade faster and yield higher RMSE because high-frequency
    # transitions (edges) are heavily discarded by the discrete cosine transform (DCT) quantisation in JPEG.

In [ ]:
# Solution 3: best_quality_under_budget using binary search
def best_quality_under_budget(image: np.ndarray, max_bytes: int) -> int:
    """Find the highest JPEG quality (1-100) that keeps file size <= max_bytes."""
    low, high = 1, 100
    best_q = 1

    while low <= high:
        mid = (low + high) // 2
        _, buf = cv2.imencode(".jpg", image, [cv2.IMWRITE_JPEG_QUALITY, mid])
        size = len(buf)

        if size <= max_bytes:
            best_q = mid
            low = mid + 1  # Try for higher quality
        else:
            high = mid - 1  # Reduce quality budget

    return best_q


# Run test code
plot_jpeg_tradeoff(load_real_image("images/objects", "coins.jpg"))
compare_degradation(smooth_img, sharp_img)
print(
    "Best quality under 15KB:",
    best_quality_under_budget(load_real_image("images/objects", "coins.jpg"), 15000),
)

### Exercises — Image I/O and Metadata
1. Extend `load_folder` to also return a dict mapping filename -> (height, width, channels).
2. Write `opencv_to_pil(image)`, the inverse of `pil_to_opencv`.
3. Research and note in markdown which EXIF tag controls image rotation, and why ignoring it causes 'sideways photo' bugs.

Use the empty cell below to work through them.


#### Solutions — Image I/O and Metadata

In [ ]:
# Solution 1: Extend load_folder to return shape metadata dict
def load_folder(
    folder_path: Path,
) -> tuple[list[np.ndarray], dict[str, tuple[int, int, int]]]:
    """Load all images in folder and return list of images + shape metadata metadata."""
    images = []
    metadata = {}
    for filepath in sorted(folder_path.glob("*")):
        if filepath.suffix.lower() in [".png", ".jpg", ".jpeg"]:
            try:
                img = safe_imread(filepath)
                images.append(img)
                metadata[filepath.name] = img.shape
            except FileNotFoundError:
                continue
    return images, metadata

In [ ]:
# Solution 2: opencv_to_pil converter
from PIL import Image


def opencv_to_pil(image: np.ndarray) -> Image.Image:
    """Convert an OpenCV BGR image to a Pillow Image instance."""
    if image.ndim == 2:
        return Image.fromarray(image)  # Grayscale
    # Convert BGR to RGB channel order for Pillow
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    return Image.fromarray(rgb)

In [ ]:
# Solution 3: EXIF Tag for rotation explanation
# The EXIF tag governing rotation is tag name "Orientation" (EXIF key code 274).
# It stores an integer (1 to 8) representing how the camera was held relative to the sensor.
# Since OpenCV's standard reader (`cv2.imread`) only parses raw pixel grids, it ignores this EXIF tag.
# Consequently, images shot in portrait mode display rotated sideways. To prevent this,
# developers must read the EXIF tag (using Pillow or libraries like piexif) and apply
# the corresponding rotation matrix manually before processing with OpenCV.

## Summary

You can choose formats deliberately and build loaders that report damaged or unsupported inputs instead of silently failing.

- **Best Practices:** Keep original files immutable, record encoding settings, correct orientation before geometry work, and log skipped files in a batch job.
- **Common Pitfalls:** Repeated JPEG re-encoding, assuming EXIF is available in OpenCV, and treating 8-bit color as the only image representation.